In [16]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

pandas version: 2.2.2
numpy version: 2.0.2


In [17]:
df = pd.read_csv('marketing_AB.csv')
print("Shape:", df.shape)
df.head()

Shape: (588101, 7)


,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [18]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nInfo:")
df.info()
print("\nNulls:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
print("\nNumeric summary:\n", df.describe())
print("\nCategorical summary:\n", df.describe(include='object'))
print("\ntest group counts:\n", df['test group'].value_counts())
print("\ntest group split %:")
print(df['test group'].value_counts(normalize=True) * 100)

Shape: (588101, 7)

Dtypes:
 Unnamed: 0        int64
user id           int64
test group       object
converted          bool
total ads         int64
most ads day     object
most ads hour     int64
dtype: object

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Unnamed: 0     588101 non-null  int64 
 1   user id        588101 non-null  int64 
 2   test group     588101 non-null  object
 3   converted      588101 non-null  bool  
 4   total ads      588101 non-null  int64 
 5   most ads day   588101 non-null  object
 6   most ads hour  588101 non-null  int64 
dtypes: bool(1), int64(4), object(2)
memory usage: 27.5+ MB

Nulls:
 Unnamed: 0       0
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64

Duplicates: 0

Numeric summary:
           Unnamed: 0       

In [19]:
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.columns.tolist())

['user_id', 'test_group', 'converted', 'total_ads', 'most_ads_day', 'most_ads_hour']


In [20]:
# converted_int
df['converted_int'] = df['converted'].astype(int)

# is_weekend
weekend_days = ['Saturday', 'Sunday']
df['is_weekend'] = df['most_ads_day'].isin(weekend_days).astype('int8')

# hour_period — ordered category
def hour_to_period(h):
    if 5 <= h < 12:
        return 'morning'
    elif 12 <= h < 17:
        return 'afternoon'
    elif 17 <= h < 21:
        return 'evening'
    else:
        return 'night'

df['hour_period'] = df['most_ads_hour'].apply(hour_to_period)

from pandas.api.types import CategoricalDtype
hour_cat = CategoricalDtype(categories=['morning','afternoon','evening','night'], ordered=True)
df['hour_period'] = df['hour_period'].astype(hour_cat)

# ads_bucket — fixed with duplicates='drop'
df['ads_bucket'] = pd.qcut(df['total_ads'], q=4, labels=['low','medium','high','very_high'], duplicates='drop')

df.head()

,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour,converted_int,is_weekend,hour_period,ads_bucket
0,1069124,ad,False,130,Monday,20,0,0,evening,very_high
1,1119715,ad,False,93,Tuesday,22,0,0,night,very_high
2,1144181,ad,False,21,Tuesday,18,0,0,evening,high
3,1435133,ad,False,355,Tuesday,10,0,0,morning,very_high
4,1015700,ad,False,276,Friday,14,0,0,afternoon,very_high


In [21]:
mem_before = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory before: {mem_before:.2f} MB")

df['test_group'] = df['test_group'].astype('category')
df['most_ads_day'] = df['most_ads_day'].astype('category')
df['converted_int'] = df['converted_int'].astype('int8')
df['is_weekend'] = df['is_weekend'].astype('int8')
df['most_ads_hour'] = df['most_ads_hour'].astype('int8')
df['total_ads'] = pd.to_numeric(df['total_ads'], downcast='integer')

mem_after = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory after: {mem_after:.2f} MB")
print(f"Reduction: {(1 - mem_after/mem_before)*100:.1f}%")

Memory before: 80.28 MB
Memory after: 10.10 MB
Reduction: 87.4%


In [22]:
print("Final shape:", df.shape)
print("\nFinal dtypes:\n", df.dtypes)
print("\nFinal nulls:\n", df.isnull().sum())
print("Final duplicates:", df.duplicated().sum())

df.to_csv('marketing_AB_cleaned.csv', index=False)
print("\n✅ Saved as marketing_AB_cleaned.csv")

Final shape: (588101, 10)

Final dtypes:
 user_id             int64
test_group       category
converted            bool
total_ads           int16
most_ads_day     category
most_ads_hour        int8
converted_int        int8
is_weekend           int8
hour_period      category
ads_bucket       category
dtype: object

Final nulls:
 user_id          0
test_group       0
converted        0
total_ads        0
most_ads_day     0
most_ads_hour    0
converted_int    0
is_weekend       0
hour_period      0
ads_bucket       0
dtype: int64
Final duplicates: 0

✅ Saved as marketing_AB_cleaned.csv
